#  **Fraud** **Detection**

## 1 Importing LIbraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 2 Importing the Dataset




In [ ]:
df=pd.read_csv('fraud.csv')
print(df.head())
print(df.shape)
print(df.info())
print(df.describe())

## 3 Data Preprocessing

The raw data has several issues we need to fix before modeling:
- Categorical columns have extra single-quote characters
- The `age` column contains `'U'` for unknown values
- `gender` has non-standard values: `E` (Enterprise) and `U` (Unknown)
- `zipcodeOri` and `zipMerchant` have only one unique value → useless for prediction
- `customer` and `merchant` are raw IDs → not useful for a general ML model

checked For any null values in the dataset and examined the distribution of the target variable (fraud) using value_counts() to understand class balance and determine whether the dataset is imbalanced

### 3.1 Checking for Null Values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Target distribution:')
print(df['fraud'].value_counts())

Created a copy of the original dataset to preserve the raw data.
Removed leading and trailing single quotation marks (') from selected categorical columns using str.strip("'")

In [ ]:
df_copy=df.copy()
cols=['customer','age','gender','zipcodeOri','merchant','zipMerchant','category']
for col in cols:
   df_copy[col] = df_copy[col].str.strip("'")

### 3.2 Handling the age

Replaced occurrences of 'U' (unknown age values) with the most frequent age value (mode) in the dataset and onverted the age column from a string type to an integer type using astype(int).

In [ ]:
print("Age unique values before fix:", df_copy['age'].unique())

age_mode = df_copy['age'].replace('U', np.nan).dropna().mode()[0]
df_copy['age'] = df_copy['age'].replace('U', age_mode).astype(int)

print(f"'U' replaced with mode: {age_mode}")
print("Age unique values after fix:", sorted(df_copy['age'].unique()))

### 3.3 Handling the Gender

In [ ]:
print("Gender distribution before fix:")
print(df_copy['gender'].value_counts())

df_copy['gender'] = df_copy['gender'].replace({'E': 'U'})

print("\nGender distribution after fix:")
print(df_copy['gender'].value_counts())

In [ ]:
print('zipcodeOri  unique:', df_copy['zipcodeOri'].nunique())
print('zipMerchant unique:', df_copy['zipMerchant'].nunique())
df_copy = df_copy.drop(columns=['zipcodeOri', 'zipMerchant'])

### 3.4 Dropping unique and identfiers columns

customer / merchant are raw IDs — useful for graph-based features in production but not for a general ML model here so we drop them

In [ ]:
df_copy = df_copy.drop(columns=['customer', 'merchant'])
print("Remaining columns:", df_copy.columns.tolist())

###3.5 Encoding Independent Variables

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd

ct = ColumnTransformer(
    transformers=[
        ('encoder',
         OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         ['age', 'gender', 'category'])
    ],
    remainder='passthrough'
)

df_encoded = ct.fit_transform(df_copy)

df_processed = pd.DataFrame(
    df_encoded,
    columns=list(ct.named_transformers_['encoder']
                 .get_feature_names_out(['age', 'gender', 'category']))
            + [col for col in df_copy.columns if col not in ['age', 'gender', 'category']]  )
df_processed.head()



In [ ]:
df_processed = df_processed.apply(pd.to_numeric, errors='coerce')
df_processed = df_processed.fillna(0)

## 4 Feature Engineering

### 4.1 Relevant Features

In [ ]:
df_processed['time_of_day'] = df_processed['step'] % 24
df_processed = df_processed.drop(columns=['step'])
df_processed.head()

### 4.2 Scaling the amount column

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_processed['amount'] = scaler.fit_transform(df_processed[['amount']])
df_processed['amount'].fillna(df_processed['amount'].mean(), inplace=True)
df_processed.head()

 ### Splitting dataset into independent variables and target

In [ ]:
X = df_processed.drop(columns=['fraud'])
y = df_processed['fraud']
print('Shape of the Dataframe:')
print(df_processed.shape)
print('Shape of X:')
print(X.shape)
print(X.isnull().sum().sum())

## 5 Exploratory Data Analysis (EDA)

Before modeling, we need to understand the data: distributions, relationships, and outliers.

 ### 5.1 Distribution of target variable (class imbalance)

In [ ]:
import matplotlib.pyplot as plt

counts = df_processed['fraud'].value_counts().sort_index()
percentages = counts / counts.sum() * 100
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar Chart
ax1 = axes[0]

bars = ax1.bar(
    ['Legitimate', 'Fraudulent'],
    counts,
    color=['#4a90e2', '#e25c5c']
)

labels = [f"{count:,}\n({pct:.1f}%)"
          for count, pct in zip(counts, percentages)]
ax1.bar_label(bars, labels=labels, padding=7)
ax1.set_title("Fraud Distribution (Bar Chart)", pad=20)
ax1.set_ylabel("Count")

# Pie Chart

ax2 = axes[1]
ax2.pie(
    counts,
    labels=['Legitimate', 'Fraudulent'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4a90e2', '#e25c5c']
)

ax2.set_title("Fraud Distribution (Pie Chart)", pad=20)
plt.tight_layout()
plt.show()

print("Severe class imbalance: fraud is only about 1.2% of all transactions.")
print("Accuracy alone is misleading; AUC-ROC and F1-score will be used for evaluation.")

###5.2 Transaction Amount Distribution
Do fraudsters spend more or less than legitimate customers?
We compare the amount distribution for fraud vs. non-fraud.

In [ ]:
df_eda = df.copy()
for col in ['customer','age','gender','zipcodeOri','merchant','zipMerchant','category']:
    df_eda[col] = df_eda[col].str.strip("'")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transaction Amount: Fraud vs. Legitimate', fontsize=13, fontweight='bold')

df_eda.boxplot(column='amount', by='fraud', ax=axes[0],
               patch_artist=True,
               boxprops=dict(facecolor='#4a90e2', color='black'),
               medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Box Plot of Amount by Class')
axes[0].set_xlabel('Fraud (0=Legitimate, 1=Fraud)')
axes[0].set_ylabel('Amount')
plt.sca(axes[0])
plt.title('Box Plot of Amount by Class')

for label, color, name in [(0, '#4a90e2', 'Legitimate'), (1, '#e25c5c', 'Fraudulent')]:
    subset = df_eda[df_eda['fraud'] == label]['amount']
    subset[subset < 2000].plot.kde(ax=axes[1], color=color, label=name, linewidth=2)
axes[1].set_title('Amount Density (clipped at 2000 for visibility)')
axes[1].set_xlabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Key observation:")
print(f"  Avg amount – Legitimate: ${df_eda[df_eda['fraud']==0]['amount'].mean():.2f}")
print(f"  Avg amount – Fraudulent: ${df_eda[df_eda['fraud']==1]['amount'].mean():.2f}")

###5.3 Fraud Rate by Category


In [ ]:
fraud_by_cat = df_eda.groupby('category')['fraud'].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(13, 5))
bars = plt.bar(fraud_by_cat.index, fraud_by_cat.values,
               color=['#e25c5c' if v > 5 else '#4a90e2' for v in fraud_by_cat.values],
               edgecolor='black')
plt.bar_label(bars, labels=[f"{v:.1f}%" for v in fraud_by_cat.values], padding=3, fontsize=9)
plt.title('Fraud Rate (%) by Merchant Category', fontsize=13, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Fraud Rate (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Top 3 highest-risk categories:")
print(fraud_by_cat.head(3))

###5.4 Fraud Rate by Gender

In [ ]:
df_eda['gender_clean'] = df_eda['gender'].replace({'E': 'U'})
fraud_by_gender = df_eda.groupby('gender_clean')['fraud'].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(7, 4))
bars = plt.bar(fraud_by_gender.index, fraud_by_gender.values,
               color=['#e25c5c', '#f5a623', '#4a90e2', '#7ed321'], edgecolor='black')
plt.bar_label(bars, labels=[f"{v:.2f}%" for v in fraud_by_gender.values], padding=3)
plt.title('Fraud Rate (%) by Gender', fontsize=12, fontweight='bold')
plt.xlabel('Gender (F=Female, M=Male, U=Unknown/Enterprise)')
plt.ylabel('Fraud Rate (%)')
plt.tight_layout()
plt.show()

###5.5 Fraud Rate by Age **Group**

In [ ]:
age_labels = {0: '≤18', 1: '19-25', 2: '26-35', 3: '36-45', 4: '46-55', 5: '56-65', 6: '>65'}
df_eda['age_num'] = df_eda['age'].replace('U', np.nan)
df_eda['age_num'] = pd.to_numeric(df_eda['age_num'], errors='coerce')

fraud_by_age = df_eda.groupby('age_num')['fraud'].mean().sort_index() * 100
fraud_by_age.index = [age_labels.get(int(i), str(i)) for i in fraud_by_age.index]

plt.figure(figsize=(9, 4))
bars = plt.bar(fraud_by_age.index, fraud_by_age.values, color='#7b68ee', edgecolor='black')
plt.bar_label(bars, labels=[f"{v:.2f}%" for v in fraud_by_age.values], padding=3)
plt.title('Fraud Rate (%) by Age Group', fontsize=12, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Fraud Rate (%)')
plt.tight_layout()
plt.show()

###5.6 Fraud by Time of Day

In [ ]:
df_copy_eda = df_processed.copy()
fraud_by_hour = df_copy_eda.groupby('time_of_day')['fraud'].mean() * 100

plt.figure(figsize=(13, 4))
plt.plot(fraud_by_hour.index, fraud_by_hour.values, marker='o', color='#e25c5c', linewidth=2)
plt.fill_between(fraud_by_hour.index, fraud_by_hour.values, alpha=0.2, color='#e25c5c')
plt.title('Fraud Rate (%) by Hour of Day', fontsize=12, fontweight='bold')
plt.xlabel('Hour of Day (0 = midnight)')
plt.ylabel('Fraud Rate (%)')
plt.xticks(range(0, 24))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

##6 Model Selection

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report

##7 Model Training

###7.1 Splitting the dataset into training and testing


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
y_train.unique()

*Evaluation metrics:** We use **AUC-ROC** and **F1-score** — NOT accuracy.
Accuracy is misleading here: a model that always predicts 'not fraud' scores 98.8% accuracy but catches zero fraud.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)

    results[name] = {'AUC-ROC': auc, 'F1': f1}

    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred))


results_df = pd.DataFrame(results).T
print('\n Comparison (No SMOTE):')
print(results_df)

Visual Comparison between the 3 models

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(results_df))
w = 0.35
bars1 = ax.bar(x - w/2, results_df['AUC-ROC'], w, label='AUC-ROC', color='#4a90e2', edgecolor='black')
bars2 = ax.bar(x + w/2, results_df['F1'],      w, label='F1 Score', color='#e25c5c', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Baseline Model Comparison (Before SMOTE)', fontweight='bold')
ax.legend()
for bar in ax.patches:
    ax.annotate(f'{bar.get_height():.4f}',
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

While Logistic Regression has a marginally higher AUC-ROC, XGBoost's superior F1-score suggests it does a better job at correctly identifying fraud cases without generating too many false positives or missing too many actual fraud cases, which is often the primary goal in fraud detection

###7.2 ROC Curves — All Baseline Models

The ROC curve shows the trade-off between:
- **True Positive Rate** (how much fraud we catch)
- **False Positive Rate** (how many legitimate transactions we wrongly flag)

The closer the curve hugs the **top-left corner**, the better.
AUC = the area under the curve. **1.0 = perfect, 0.5 = random guessing.**

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(9, 6))

plot_colors = {
    'Logistic Regression': '#7b68ee',
    'Random Forest':       '#f5a623',
    'XGBoost':             '#4a90e2'
}

for name, color in plot_colors.items():
    prob = models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{name} (AUC={auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves – Baseline Models (Before SMOTE)', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

###7.3 Confusion Matrix for XGBoost

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

xgboost_model = models['XGBoost']

y_pred_xgboost = xgboost_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_xgboost)
class_labels = ['Legitimate (0)', 'Fraudulent (1)']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - XGBoost (Before SMOTE)')
plt.show()
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correct legit) : {tn:,}')
print(f'False Positives (false alarms)   : {fp:,}')
print(f'False Negatives (missed fraud)   : {fn:,}  ← this is what we want to reduce')
print(f'True Positives  (caught fraud)   : {tp:,}')
print(f'\nFraud Recall (% of fraud caught) : {tp/(tp+fn)*100:.1f}%')

##8 Model Interpretation

In [ ]:
importance = pd.Series(
    models['XGBoost'].feature_importances_,
    index=X.columns
).sort_values(ascending=True)

top3 = importance.nlargest(3).index
colors = ['#e25c5c' if col in top3 else '#4a90e2' for col in importance.index]

plt.figure(figsize=(8, 5))
bars = plt.barh(importance.index, importance.values, color=colors, edgecolor='black')
plt.title(
    'Feature Importance – XGBoost \n(red = top 3 most important features)',
    fontsize=12, fontweight='bold'
)
plt.xlabel('Importance Score (Gain)')
plt.tight_layout()
plt.show()

print('Top 5 features driving fraud detection:')
print(importance.sort_values(ascending=False).head(5).to_string())

##9 Handling Class Imbalance with SMOTE
We saw above that the model misses a portion of fraud cases. The root cause is the **severe class imbalance** — only 1.2% of transactions are fraud.

### Why not just use the imbalanced data?
The model sees ~82 legitimate transactions for every 1 fraud sample in training. It learns to heavily favor predicting 'legitimate' because that's almost always correct. It pays little attention to the fraud minority.

**SMOTE** was selected because it balances the dataset by creating synthetic fraud examples, preserving all legitimate transactions and reducing overfitting compared to simple oversampling or undersampling.
**# ONLY applied to training data — test set stays untouched!
**

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy=0.1, random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Class distribution BEFORE SMOTE (training set):')
print(y_train.value_counts().to_dict())

print('\nClass distribution AFTER SMOTE (training set):')
print(pd.Series(y_train_sm).value_counts().to_dict())

print(f'\nTraining set grew from {len(y_train):,} → {len(y_train_sm):,} rows')
print('Test set untouched — reflects real-world distribution')

In [ ]:
neg = (y_train_sm == 0).sum()
pos = (y_train_sm == 1).sum()

xgb_smote = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

xgb_smote.fit(X_train_sm, y_train_sm)

y_pred_sm = xgb_smote.predict(X_test)
y_prob_sm = xgb_smote.predict_proba(X_test)[:, 1]

auc_sm = roc_auc_score(y_test, y_prob_sm)
f1_sm  = f1_score(y_test, y_pred_sm)

print('XGBoost + SMOTE — Test Set Performance:')
print(f'  AUC-ROC : {auc_sm:.4f}')
print(f'  F1 Score: {f1_sm:.4f}')
print()
print(classification_report(y_test, y_pred_sm, target_names=['Legitimate', 'Fraudulent']))

In [ ]:
compare = pd.DataFrame({
    'Metric':           ['AUC-ROC', 'F1 Score'],
    'XGBoost (No SMOTE)': [results['XGBoost']['AUC-ROC'], results['XGBoost']['F1']],
    'XGBoost + SMOTE':    [round(auc_sm, 4), round(f1_sm, 4)]
})
print('XGBoost Before vs After SMOTE:')
print(compare.to_string(index=False))

cm_sm = confusion_matrix(y_test, y_pred_sm)
disp_sm = ConfusionMatrixDisplay(confusion_matrix=cm_sm, display_labels=['Legitimate(0)', 'Fraudulent(1)'])

fig, ax = plt.subplots(figsize=(6, 5))

disp_sm.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix – XGBoost + SMOTE', fontweight='bold')
plt.tight_layout()
plt.show()

tn2, fp2, fn2, tp2 = cm_sm.ravel()
print(f'False Negatives (missed fraud) before SMOTE: {fn:,}')
print(f'False Negatives (missed fraud) after  SMOTE: {fn2:,}')
print(f'\nFraud Recall before SMOTE: {tp/(tp+fn)*100:.1f}%')
print(f'Fraud Recall after  SMOTE: {tp2/(tp2+fn2)*100:.1f}%')

## 10 Fine-Tuning and Optimization
XGBoost + SMOTE model is strong. Now we optimize its hyperparameters to squeeze out even better performance.
We use **RandomizedSearchCV** instead of GridSearchCV because:
- GridSearchCV tests **every possible combination** → exponentially slow with 7 parameters
- RandomizedSearchCV **randomly samples the space** → finds near-optimal parameters much faster
- With `n_iter=20` and `cv=3`, we try 20 random combinations with 3-fold cross-validation each

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 4, 5, 6],
    'learning_rate'   : [0.05, 0.1, 0.2],
    'subsample'       : [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma'           : [0, 0.1, 0.2]
}

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_sm, y_train_sm)

print(f'\n Best cross-validation AUC-ROC: {random_search.best_score_:.4f}')
print('\nBest parameters found:')
for k, v in random_search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
best_xgb = random_search.best_estimator_

y_pred_tuned = best_xgb.predict(X_test)
y_prob_tuned = best_xgb.predict_proba(X_test)[:, 1]

auc_tuned = roc_auc_score(y_test, y_prob_tuned)
f1_tuned  = f1_score(y_test, y_pred_tuned)

print('Tuned XGBoost + SMOTE — Final Test Performance:')
print(f'  AUC-ROC : {auc_tuned:.4f}')
print(f'  F1 Score: {f1_tuned:.4f}')
print()
print(classification_report(y_test, y_pred_tuned, target_names=['Legitimate', 'Fraudulent']))

In [ ]:

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
disp_tuned = ConfusionMatrixDisplay(
    confusion_matrix=cm_tuned,
    display_labels=['Legitimate', 'Fraudulent']
)
fig, ax = plt.subplots(figsize=(6, 5))
disp_tuned.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix – Tuned XGBoost + SMOTE', fontweight='bold')
plt.tight_layout()
plt.show()

tn_t, fp_t, fn_t, tp_t = cm_tuned.ravel()
print(f'False Negatives (missed fraud) : {fn_t:,}  ← minimized after tuning')
print(f'Fraud Recall                   : {tp_t/(tp_t+fn_t)*100:.1f}%')

In [ ]:
plt.figure(figsize=(9, 6))

for name, color in plot_colors.items():
    prob = models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{name} baseline (AUC={roc_auc_score(y_test,prob):.4f})')

fpr_sm, tpr_sm, _ = roc_curve(y_test, y_prob_sm)
plt.plot(fpr_sm, tpr_sm, color='#f5a623', linewidth=2, linestyle=':',
         label=f'XGBoost + SMOTE (AUC={auc_sm:.4f})')

fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_tuned)
plt.plot(fpr_t, tpr_t, color='#e25c5c', linewidth=3, linestyle='--',
         label=f'XGBoost + SMOTE + Tuning (AUC={auc_tuned:.4f}) ← BEST')

plt.plot([0,1],[0,1],'k--', linewidth=1, label='Random (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves – All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

##11 Final Model

In [ ]:
final_df = pd.DataFrame([
    {'Model': 'Logistic Regression ',    'AUC-ROC': round(results['Logistic Regression']['AUC-ROC'],4), 'F1 Score': round(results['Logistic Regression']['F1'],4)},
    {'Model': 'Random Forest ',           'AUC-ROC': round(results['Random Forest']['AUC-ROC'],4),       'F1 Score': round(results['Random Forest']['F1'],4)},
    {'Model': 'XGBoost ',                 'AUC-ROC': round(results['XGBoost']['AUC-ROC'],4),             'F1 Score': round(results['XGBoost']['F1'],4)},
    {'Model': 'XGBoost + SMOTE',                   'AUC-ROC': round(auc_sm,4),                                    'F1 Score': round(f1_sm,4)},
    {'Model': 'XGBoost + SMOTE + Tuning ',  'AUC-ROC': round(auc_tuned,4),                                 'F1 Score': round(f1_tuned,4)},
]).set_index('Model')

print('=' * 58)
print('  FINAL MODEL SCORECARD')
print('=' * 58)
print(final_df.to_string())
print('=' * 58)
print(f'\n Best Model : XGBoost + SMOTE + Hyperparameter Tuning')
print(f'   AUC-ROC    : {auc_tuned:.4f}')
print(f'   F1 Score   : {f1_tuned:.4f}')

